<a href="https://colab.research.google.com/github/MichalSlowakiewicz/Hybrid-Graph-Neural-Networks/blob/master/core.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import Sequential, Linear, ReLU
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.nn import GCNConv, GATConv, GINConv
from sklearn.metrics import roc_auc_score

# ==========================================
# 1. POBRANIE DANYCH I NEGATIVE SAMPLING
# ==========================================
print("Pobieranie zbioru danych Cora (ponad 2700 węzłów, 5000+ krawędzi)...")
dataset = Planetoid(root='data/Cora', name='Cora')
data = dataset[0]

# Genialne narzędzie z PyG: Automatycznie ukrywa część krawędzi do testów
# i generuje fałszywe krawędzie (Negative Sampling) w proporcji 1:1!
transform = T.RandomLinkSplit(
    num_val=0.1,   # 10% krawędzi do walidacji
    num_test=0.2,  # 20% krawędzi ukrywamy do finałowego testu
    is_undirected=True,
    add_negative_train_samples=True # Nasz Negative Sampling z prezentacji!
)
train_data, val_data, test_data = transform(data)
print(f"Dane podzielone! Krawędzie treningowe: {train_data.edge_label.size(0)}")


# ==========================================
# 2. DEFINICJE MODELI (ENKODERY + DEKODER)
# ==========================================
# Dekoder (wspólny dla wszystkich) - ocenia prawdopodobieństwo istnienia krawędzi
def decode(z, edge_label_index):
    # Iloczyn skalarny między dwoma wektorami (im bardziej podobne, tym większa szansa na krawędź)
    return (z[edge_label_index[0]] * z[edge_label_index[1]]).sum(dim=-1)

# Model 1: Klasyczny GCN (Uśrednianie)
class GCN_Net(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)

    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)

# Model 2: Klasyczny GAT (Atencja)
class GAT_Net(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=2, concat=False)
        self.conv2 = GATConv(hidden_channels, hidden_channels, heads=2, concat=False)

    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)

# Model 3: Nasza Gwiazda - HYBRYDA (GAT -> GIN)
class Hybrid_GAT_GIN_Net(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        # Warstwa 1: GAT (Odfiltrowanie szumu za pomocą atencji)
        self.conv1 = GATConv(in_channels, hidden_channels, heads=1)

        # Warstwa 2: GIN (Zrozumienie struktury za pomocą sieci MLP)
        mlp = Sequential(Linear(hidden_channels, hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINConv(mlp)

    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)


# ==========================================
# 3. FUNKCJA TRENUJĄCA
# ==========================================
def train_and_test(model, name):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = torch.nn.BCEWithLogitsLoss() # Klasyfikacja binarna (jest link / nie ma linku)

    print(f"\n--- Rozpoczynamy trening modelu: {name} ---")

    # Szybki trening na 50 epok
    for epoch in range(1, 51):
        model.train()
        optimizer.zero_grad()

        # Enkodowanie węzłów i dekodowanie krawędzi treningowych
        z = model.encode(train_data.x, train_data.edge_index)
        out = decode(z, train_data.edge_label_index)

        loss = criterion(out, train_data.edge_label.float())
        loss.backward()
        optimizer.step()

    # Ewaluacja na ukrytym zbiorze TESTOWYM
    model.eval()
    with torch.no_grad():
        z = model.encode(test_data.x, test_data.edge_index)
        out = decode(z, test_data.edge_label_index).sigmoid()

        # Liczymy metrykę AUC (Im bliżej 1.0, tym lepiej)
        auc = roc_auc_score(test_data.edge_label.cpu().numpy(), out.cpu().numpy())
        print(f"Wynik AUC na zbiorze testowym: {auc:.4f}")
        return auc

# ==========================================
# 4. URUCHOMIENIE EKSPERYMENTU
# ==========================================
in_channels = dataset.num_features
hidden_channels = 64

model_gcn = GCN_Net(in_channels, hidden_channels)
model_gat = GAT_Net(in_channels, hidden_channels)
model_hybrid = Hybrid_GAT_GIN_Net(in_channels, hidden_channels)

auc_gcn = train_and_test(model_gcn, "GCN (Homogeniczny)")
auc_gat = train_and_test(model_gat, "GAT (Homogeniczny z Atencją)")
auc_hybrid = train_and_test(model_hybrid, "HYBRYDA (GAT-GIN)")

print("\n=== PODSUMOWANIE WYNIKÓW (Link Prediction) ===")
print(f"GCN:     {auc_gcn:.4f}")
print(f"GAT:     {auc_gat:.4f}")
print(f"HYBRYDA: {auc_hybrid:.4f}")